# 06 · Demo Gradio

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tomas-pucv/Camanchaca-Predict/blob/main/notebooks/06_demo.ipynb)

**Propósito.** Lanzar la demo (visibilidad MOR, banda, incertidumbre entre semillas, velocidad
ILUSTRATIVA, Grad-CAM/rollout, avisos fuera de distribución y simulación de niebla) con un enlace
público, y grabar el video de respaldo.

**Tiempo esperado.** ~5 min (+ grabación). **Entorno.** CPU basta (GPU opcional).
**Salidas.** Enlace `*.gradio.live` (dura hasta 1 semana mientras la sesión viva) y
`MyDrive/camanchaca/demo_respaldo.mp4`.

Para la presentación conviene correr la demo **en el portátil** (`python app/app.py --exp main`,
ver `app/README.md`) y usar Colab solo como respaldo.

## Configuración común
Misma celda en todos los notebooks (detalles en `00_setup_colab`). En **Colab**: monta Drive,
fija `CAMANCHACA_DATA=/content/data` y `CAMANCHACA_RUNS=/content/drive/MyDrive/camanchaca/runs`,
clona/actualiza el repo en la referencia `REF`, instala el paquete (`pip install -e ".[demo,data]"`,
**sin** reinstalar torch) y desempaqueta los `.tar` de datos de Drive (salvo `NEEDS_DATA = False`). **Fuera de Colab** (`SMOKE=True`)
usa datos sintéticos y pesos aleatorios generados por `camanchaca smoke`: sirve para verificar que
todo corre, **sus números no son resultados**.

In [ ]:
NEEDS_DATA = False   # la demo no usa las imágenes de entrenamiento: no desempaquetar los .tar

In [ ]:
# Configuración común (idéntica en los notebooks 00-06)
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = not IN_COLAB   # True: datos sintéticos + pesos aleatorios (solo verifica que todo corre)
REF = "main"           # rama/tag/commit del repo; para la entrega final usar "v1.0-final"
REPO_URL = "https://github.com/Tomas-pucv/Camanchaca-Predict.git"
DRIVE_DIR = Path("/content/drive/MyDrive/camanchaca")   # tars de datos, corridas, pesos y credenciales

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["CAMANCHACA_DATA"] = "/content/data"
    os.environ["CAMANCHACA_RUNS"] = str(DRIVE_DIR / "runs")
    REPO = Path("/content/Camanchaca-Predict")
    if not REPO.exists():
        !git clone -q {REPO_URL} {REPO}
    !git -C {REPO} fetch -q --all --tags && git -C {REPO} checkout -q {REF} && (git -C {REPO} pull -q --ff-only 2>/dev/null || true)
    %cd {REPO}
    !pip install -q -e ".[demo,data]"   # NO reinstala torch/torchvision: se usan los de Colab
    if not SMOKE and globals().get("NEEDS_DATA", True):
        for tar in sorted((DRIVE_DIR / "data").glob("*.tar")):   # imágenes preprocesadas (notebook 01)
            !camanchaca unpack --tar {tar}
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    %cd {REPO}
    if shutil.which("camanchaca") is None:
        print("AVISO: falta el comando `camanchaca`: ejecutar  pip install -e '.[dev,demo]'")

WORK = Path(os.environ.get("CAMANCHACA_SMOKE_DIR", str(REPO / "local_workspace" / "nb_smoke")))
if SMOKE:
    os.environ.update(CAMANCHACA_DATA=str(WORK / "data"), CAMANCHACA_RUNS=str(WORK / "runs"),
                      CAMANCHACA_REPORTS=str(WORK / "reports"), HF_HUB_OFFLINE="1")
    if not (WORK / "smoke.yaml").exists():   # ~4-6 min en CPU, una sola vez
        !camanchaca smoke --workdir {WORK}

sys.path.insert(0, str(REPO / "src"))   # importable sin reiniciar el kernel tras `pip install -e`
from camanchaca.config import load_yaml
from camanchaca.paths import data_root, reports_dir, runs_root

EXP = str(WORK / "smoke.yaml") if SMOKE else "configs/experiment/main.yaml"
EXP_NAME = load_yaml(EXP)["name"]
print(f"Colab={IN_COLAB} | SMOKE={SMOKE} | EXP={EXP}")
print(f"datos={data_root()} | corridas={runs_root()} | reportes={reports_dir()}")

## 1. Pesos
Usa los mejores checkpoints de `02_train` (en Drive). Si no están, los descarga del GitHub
Release con `scripts/fetch_weights.py` (verifica sha256) o los copia de `MyDrive/camanchaca/weights`.

In [ ]:
from camanchaca.demo.app import discover_checkpoints
CKPTS = discover_checkpoints(EXP, runs_root())
if not CKPTS and not SMOKE:
    !{sys.executable} scripts/fetch_weights.py --exp {EXP_NAME} --drive-dir {DRIVE_DIR}/weights
    CKPTS = discover_checkpoints(EXP, runs_root())
print(len(CKPTS), "checkpoints:", *CKPTS, sep="\n")

## 2. Lanzar
En Colab muestra la interfaz en la celda y un enlace público `*.gradio.live` (`share=True`).
Con `USE_CLI = True` se lanza en cambio `python app/app.py --exp ... --share` (bloquea la celda
hasta detenerla con ■).

In [ ]:
USE_CLI = False   # True: misma demo vía `app/app.py` (bloquea la celda)
if USE_CLI and not SMOKE:
    !{sys.executable} app/app.py --exp {EXP} --runs-root {runs_root()} --share
else:
    from camanchaca.demo.app import launch_demo, load_predictors
    PREDICTORS = load_predictors(exp=EXP, runs_root=runs_root())
    demo = launch_demo(PREDICTORS, share=IN_COLAB, prevent_thread_lock=True, quiet=SMOKE)

## 3. Video de respaldo (2–3 min)
Guion sugerido:
1. Foto de La Pólvora con camanchaca → V (MOR), banda, incertidumbre (×) y Grad-CAM −ŷ (30 s).
2. Misma foto con DeiT-S + attention rollout: diferencias con ResNet-50 (30 s).
3. *Simular niebla* sobre una foto despejada con V = 50, 150 y 500 m: V fijada vs V̂ (45 s).
4. Un caso fuera de distribución (foto nocturna) → aviso (15 s).
5. *Acerca de*: supuestos, velocidad ILUSTRATIVA y "no apto para uso operativo" (15 s).

Grabar la pantalla (OBS, Xbox Game Bar o QuickTime), guardar en
`MyDrive/camanchaca/demo_respaldo.mp4` y enlazarlo en el README.

In [ ]:
if SMOKE and "demo" in globals():   # en Colab la demo sigue activa; para detenerla: demo.close()
    demo.close()